**MAESTRÍA EN ECONOMÍA APLICADA - UBA 2025**
**TALLER DE PROGRAMACIÓN**
**GRUPO 2**

# PROYECTO FINAL: TALLER DE PROGRAMACIÓN

In [6]:
pip install pandas numpy scipy scikit-learn tslearn hdbscan matplotlib seaborn geopandas folium plotly openpyxl tqdm


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: pandas in c:\users\marti\anaconda3\lib\site-packages (2.2.3)
  Using cached hdbscan-0.8.40.tar.gz (6.9 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached geopandas-1.1.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached folium-0.20.0-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached pyogrio-0.12.1-cp313-cp313-win_amd64.whl.metadata (6.0 kB)
  Using cached pyproj-3.7.2-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached shapely-2.1.2-cp313-cp313-win_amd64.whl.metadata (7.1 kB)
  Using cached branca-0.8.2-py3-none-any.whl.metadata (1.7 kB)
Using cached geopandas-1.1.1-py3-none-any.whl (338 k

  error: subprocess-exited-with-error
  
  × Building wheel for hdbscan (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [39 lines of output]
      C:\Users\marti\AppData\Local\Temp\pip-build-env-q7sktyl7\overlay\Lib\site-packages\setuptools\_distutils\dist.py:289: UserWarning: Unknown distribution option: 'test_suite'
        warnings.warn(msg)
      C:\Users\marti\AppData\Local\Temp\pip-build-env-q7sktyl7\overlay\Lib\site-packages\setuptools\_distutils\dist.py:289: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      C:\Users\marti\AppData\Local\Temp\pip-build-env-q7sktyl7\overlay\Lib\site-packages\setuptools\dist.py:759: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: 

1) IMPORTS + CONFIGURACIÓN GENERAL

In [7]:
import os, glob, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm

# Preproc y ML
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import silhouette_score
from scipy import stats

# DTW / clustering
from tslearn.clustering import TimeSeriesKMeans
from tslearn.metrics import cdist_dtw

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")


In [3]:
BASE_PATH = r"C:\Users\marti\Downloads\Bases"  # ajustá si hace falta
START_YEAR = 2019
END_YEAR = 2025


2) FUNCIONES PARA CARGA DE BASES MENSUALES

In [4]:
def cargar_bases(base_path=BASE_PATH, start_year=START_YEAR, end_year=END_YEAR):
    dfs = []
    for year in range(start_year, end_year + 1):
        folder_year = os.path.join(base_path, str(year))
        if not os.path.exists(folder_year):
            continue
        # buscar carpetas StockArgenprop_*
        posibles = glob.glob(os.path.join(folder_year, "StockArgenprop_*"))
        for item in posibles:
            # si es carpeta, buscar archivos dentro
            if os.path.isdir(item):
                archivos = glob.glob(os.path.join(item, "*"))
            else:
                archivos = [item]

            for f in archivos:
                ext = os.path.splitext(f)[1].lower()
                try:
                    if ext == ".csv":
                        df = pd.read_csv(f, low_memory=False, encoding="utf-8")
                    elif ext in [".xls", ".xlsx"]:
                        df = pd.read_excel(f)
                    else:
                        continue
                    df["ArchivoOrigen"] = os.path.basename(f)
                    dfs.append(df)
                except Exception as e:
                    print("Error leyendo", f, "->", e)
    if len(dfs) == 0:
        raise ValueError("No se cargaron archivos. Verificá rutas y extensiones.")
    df_total = pd.concat(dfs, ignore_index=True)
    print("Total archivos cargados:", len(dfs))
    print("Total filas concatenadas:", df_total.shape[0])
    return df_total


In [5]:
df_raw = cargar_bases()


KeyboardInterrupt: 

3) LIMPIEZA + GENERACIÓN DE VARIABLES


In [ ]:
def limpiar_y_generar(df):
    # Homologar nombres (por si hay columnas con espacios o mayúsculas)
    cols_map = {c: c.strip() for c in df.columns}
    df.rename(columns=cols_map, inplace=True)

    # Fechas
    for col in ["FechaPublicacion", "FechaModificacion"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # month a partir de FechaModificacion cuando exista, si no FechaPublicacion
    if "FechaModificacion" in df.columns:
        df["month"] = df["FechaModificacion"].dt.to_period("M")
    else:
        df["month"] = df["FechaPublicacion"].dt.to_period("M")

    # Variables numéricas
    num_cols = ["MontoOperacion", "PropiedadSuperficieTotal", "SuperficieCubierta",
                "SuperficieDescubierta", "CantidadAmbientes", "CantidadDormitorios",
                "Antiguedad", "DepartamentoExpensas", "CantidadCocheras",
                "Latitud", "Longitud"]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Precio por m2 (usar SuperficieCubierta o PropiedadSuperficieTotal)
    df["superficie_ref"] = df["PropiedadSuperficieTotal"].fillna(df.get("SuperficieCubierta"))
    df["precio_m2"] = df["MontoOperacion"] / df["superficie_ref"]
    # Manejo de casos extremos
    df.loc[df["precio_m2"] <= 0, "precio_m2"] = np.nan

    # Normalizaciones básicas (puede haber columnas de texto)
    # Mantener Codigo como str
    if "Codigo" in df.columns:
        df["Codigo"] = df["Codigo"].astype(str)

    return df

df = limpiar_y_generar(df_raw)


4. ESTADÍSTICA DESCRIPTIVA

In [ ]:
# Estadística básica global
desc_global = df[["MontoOperacion","superficie_ref","precio_m2","CantidadAmbientes","CantidadDormitorios"]].describe().T
desc_global["missing_pct"] = df[["MontoOperacion","superficie_ref","precio_m2","CantidadAmbientes","CantidadDormitorios"]].isna().mean().values
desc_global

# Conteos por mes (tamaño de muestra mensual)
conteo_mes = df.groupby("month").size().rename("n_obs").reset_index()
conteo_mes.head()

# Distribución por TipoPropiedad y TipoOperacion
if "TipoPropiedad" in df.columns:
    tipo_prop_counts = df["TipoPropiedad"].value_counts().reset_index().rename(columns={"index":"TipoPropiedad","TipoPropiedad":"count"})
else:
    tipo_prop_counts = pd.DataFrame()

# Precio_m2 por barrio (top 20)
if "Barrio" in df.columns:
    precio_barrio = df.groupby("Barrio")["precio_m2"].median().dropna().sort_values(ascending=False).head(20)
else:
    precio_barrio = pd.Series()


In [ ]:
desc_global.to_csv("desc_global.csv")
conteo_mes.to_csv("conteo_mes.csv", index=False)
precio_barrio.to_csv("precio_barrio_top20.csv")


5. CONSTRUIR PANEL MENSUAL

In [ ]:
# 1) Filtrar periodo seleccionado (ejemplo 2019-01 en adelante)
df = df[df["month"].notna()]
df = df[df["month"] >= pd.Period(f"{START_YEAR}-01", freq="M")]

# 2) Agrupar por Barrio x month. Tomamos mediana de precio_m2, counts y mediana superficie
group_cols = ["Barrio","month"]
panel = df.groupby(group_cols).agg(
    n_listings = ("Codigo","count"),
    precio_m2_med = ("precio_m2","median"),
    superficie_med = ("superficie_ref","median")
).reset_index()

# 3) Pivot para obtener series: filas=Barrio, columnas=month
pivot = panel.pivot(index="Barrio", columns="month", values="precio_m2_med")
print("Panel shape (barrio x month):", pivot.shape)


6. ELIMINACIÓN DE OUTLIERS

In [ ]:
def eliminar_outliers_por_mes(df, var="precio_m2", low_q=0.01, high_q=0.99):
    dfs = []
    for m, g in df.groupby("month"):
        low = g[var].quantile(low_q)
        high = g[var].quantile(high_q)
        gf = g[(g[var].isna()) | ((g[var] >= low) & (g[var] <= high))]
        dfs.append(gf)
    return pd.concat(dfs, ignore_index=True)

df_clean = eliminar_outliers_por_mes(df, "precio_m2", 0.01, 0.99)

# Reconstruir panel con df_clean
panel_clean = df_clean.groupby(["Barrio","month"]).agg(
    n_listings=("Codigo","count"),
    precio_m2_med=("precio_m2","median")
).reset_index()
pivot_clean = panel_clean.pivot(index="Barrio", columns="month", values="precio_m2_med")


In [ ]:
min_coverage = 0.6  # exigir 60% de meses con dato
valid_series = pivot_clean.index[(pivot_clean.notna().mean(axis=1) >= min_coverage)].tolist()
pivot_final = pivot_clean.loc[valid_series]
print("Series válidas (Barrio):", len(pivot_final))


In [ ]:
pivot_imputed = pivot_final.copy().T  # months x barrios for interpolation
pivot_imputed = pivot_imputed.interpolate(method="linear", limit_direction="both", axis=0)
pivot_imputed = pivot_imputed.fillna(method="ffill").fillna(method="bfill")
pivot_imputed = pivot_imputed.T  # barrios x months


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_ts = scaler.fit_transform(pivot_imputed.values)  # cada fila es una serie
# Reshape para tslearn: (n_series, time_len, 1)
X_ts3 = X_ts.reshape((X_ts.shape[0], X_ts.shape[1], 1))


7. CLUSTERING DTW — PRIMER NIVEL

In [ ]:
# Calcular matriz de distancias DTW (opcional: solo para silhouette)
# ADVERTENCIA: cdist_dtw puede ser costoso si hay muchas series.
dist_matrix = cdist_dtw(X_ts3.squeeze())  # shape (n_series, n_series)

# Función para probar varios k y calcular silhouette (usa dist_matrix precomputada)
def silhouette_dtw(dist_matrix, labels):
    # silhouette_score acepta metric='precomputed' when you pass distance matrix and labels
    return silhouette_score(dist_matrix, labels, metric="precomputed")

from sklearn.cluster import KMeans

results_k = []
K_RANGE = range(2,9)
for k in K_RANGE:
    km_dtw = TimeSeriesKMeans(n_clusters=k, metric="dtw", max_iter=20, random_state=0)
    labels_k = km_dtw.fit_predict(X_ts3)
    sil = silhouette_dtw(dist_matrix, labels_k)
    results_k.append((k, sil))
    print("k=",k,"silhouette (DTW):", sil)

# Elegir k con mayor silhouette
best_k = max(results_k, key=lambda x: x[1])[0]
print("Best k (primer nivel):", best_k)


In [ ]:
model_lvl1 = TimeSeriesKMeans(n_clusters=best_k, metric="dtw", max_iter=50, random_state=0)
labels_lvl1 = model_lvl1.fit_predict(X_ts3)
pivot_imputed["cluster_lvl1"] = labels_lvl1


In [ ]:
time_index = pivot_imputed.columns[:-1]  # months
plt.figure(figsize=(10,6))
for i in range(best_k):
    prototype = model_lvl1.cluster_centers_[i].squeeze()
    plt.plot(prototype, label=f"Cluster {i}")
plt.legend()
plt.title("Prototipos DTW - Nivel 1 (precio m2)")
plt.xlabel("Meses (orden temporal)")
plt.ylabel("precio_m2 (estandarizado)")
plt.show()


8. CLUSTERING DTW — SEGUNDO NIVEL

In [ ]:
subcluster_labels = {}
for c in np.unique(labels_lvl1):
    idx = np.where(labels_lvl1 == c)[0]
    X_sub = X_ts3[idx]
    if len(idx) < 3:
        subcluster_labels[c] = np.zeros(len(idx), dtype=int)
        continue
    # elegir k_sub (por ejemplo 2 o probar con silhouette)
    k_sub = 2
    model_sub = TimeSeriesKMeans(n_clusters=k_sub, metric="dtw", max_iter=50, random_state=0)
    labels_sub = model_sub.fit_predict(X_sub)
    subcluster_labels[c] = labels_sub
    print(f"Cluster {c} -> subclusters: {np.unique(labels_sub)}")


Estrategia B (clusterizar los prototipos del nivel1 si querés agrupar clusters similares):

In [ ]:
# Centroides nivel 1: shape (best_k, time_len, 1)
centroids = model_lvl1.cluster_centers_
# reclusterizar estos centroides en K2 grupos (ej K2=3)
K2 = 3
model_lvl2 = TimeSeriesKMeans(n_clusters=K2, metric="dtw", max_iter=50, random_state=0)
labels_lvl2 = model_lvl2.fit_predict(centroids)
print("Labels nivel2 para cada prototipo nivel1:", labels_lvl2)


9. EVALUACIÓN Y MÉTRICAS

In [ ]:
# Tamaño
import collections
count_by_cluster = collections.Counter(labels_lvl1)
print("Tamaños nivel 1:", count_by_cluster)

# Evolución promedio por cluster (prototipos ya estandarizados)
for i in range(best_k):
    plt.plot(model_lvl1.cluster_centers_[i].squeeze(), label=f"Cluster {i}")
plt.legend(); plt.title("Evolución promedio (prototipos)"); plt.show()


10. GUARDAR RESULTADOS / TABLAS PARA PRESENTACIÓN

In [ ]:
# DataFrame con barrio, cluster y stats
df_clusters = pivot_imputed.reset_index().rename(columns={"index":"Barrio"})
df_clusters["cluster_lvl1"] = df_clusters["cluster_lvl1"].astype(int)
# Agregar algunas estadísticas descriptivas por cluster (mediana precio, trend slope)
def slope(series):
    # slope simple linea sobre tiempo (series es numpy array)
    x = np.arange(len(series))
    mask = ~np.isnan(series)
    if mask.sum() < 2: return np.nan
    m, b = np.polyfit(x[mask], series[mask], 1)
    return m
stats_by_barrio = []
for _, row in df_clusters.iterrows():
    serie = row.iloc[1:-1].values  # ajustar indices si es necesario
    stats_by_barrio.append({
        "Barrio": row["Barrio"],
        "cluster_lvl1": row["cluster_lvl1"],
        "mean_pr": np.nanmean(serie),
        "slope_pr": slope(serie)
    })
df_stats_barrio = pd.DataFrame(stats_by_barrio)
df_stats_barrio.groupby("cluster_lvl1").agg({"Barrio":"count","mean_pr":"median","slope_pr":"median"}).to_csv("cluster_summary.csv")
